In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_tenants = spark.table("voyage.bronze.tenants")

df_tenants.printSchema()

display(df_tenants)

In [0]:
tenant_window = (
    Window
    .partitionBy("tenant_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_airbyte_extracted_at").desc(),
    )
)

silver_tenants = (
    df_tenants
    .withColumn("_row_number", F.row_number().over(tenant_window))
    .filter(F.col("_row_number") == 1)
    .select(
        F.col("tenant_id").cast("string"),
        F.col("tenant_name").cast("string"),
        F.col("country").cast("string"),
        F.col("currency").cast("string"),
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),
    )
)

In [0]:
silver_tenants.printSchema()
display(silver_tenants)

In [0]:
(
    silver_tenants.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.silver.dim_tenant")
)

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT tenant_id) AS distinct_tenants
FROM voyage.silver.dim_tenant;

In [0]:
%sql
SELECT *
FROM voyage.silver.dim_tenant
ORDER BY tenant_id;

In [0]:
df_properties = spark.table("voyage.bronze.properties")

property_window = (
    Window
    .partitionBy("property_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_airbyte_extracted_at").desc(),
    )
)

silver_properties = (
    df_properties
    .withColumn("_row_number", F.row_number().over(property_window))
    .filter(F.col("_row_number") == 1)
    .select(
        F.col("property_id").cast("string"),
        F.col("tenant_id").cast("string"),
        F.col("property_name").cast("string"),
        F.col("property_type").cast("string"),
        F.col("destination").cast("string"),
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),
    )
)

(
    silver_properties.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.silver.dim_property")
)

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT property_id) AS distinct_properties
FROM voyage.silver.dim_property;

In [0]:
%sql
SELECT
    p.property_id,
    p.property_name,
    p.tenant_id,
    t.tenant_name
FROM voyage.silver.dim_property p
LEFT JOIN voyage.silver.dim_tenant t
    ON p.tenant_id = t.tenant_id
ORDER BY p.property_id;

In [0]:
df_customers = spark.table("voyage.bronze.customers")

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_airbyte_extracted_at").desc(),
    )
)

silver_customers = (
    df_customers
    .withColumn("_row_number", F.row_number().over(customer_window))
    .filter(F.col("_row_number") == 1)
    .select(
        F.col("customer_id").cast("string"),
        F.col("tenant_id").cast("string"),
        F.col("country").cast("string"),
        F.col("customer_type").cast("string"),
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),
    )
)
print(
    f"Silver customers: "
    f"{silver_customers.count()}"
)

print(
    f"Distinct customers: "
    f"{silver_customers.select('customer_id').distinct().count()}"
)

In [0]:
(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.silver.dim_customer")
)

In [0]:
df_bookings = spark.table("voyage.bronze.bookings")
booking_window = (
    Window
    .partitionBy("booking_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_airbyte_extracted_at").desc(),
    )
)

silver_bookings = (
    df_bookings
    .withColumn("_row_number", F.row_number().over(booking_window))
    .filter(F.col("_row_number") == 1)
    .select(
        F.col("booking_id").cast("string"),
        F.col("tenant_id").cast("string"),
        F.col("property_id").cast("string"),
        F.col("customer_id").cast("string"),
        F.to_date("booking_date").alias("booking_date"),
        F.to_date("arrival_date").alias("arrival_date"),
        F.to_date("departure_date").alias("departure_date"),
        F.col("adults").cast("int").alias("adults"),
        F.col("children").cast("int").alias("children"),
        F.col("room_nights").cast("int").alias("room_nights"),
        F.col("adr").cast("decimal(12,2)").alias("adr"),
        F.col("market_segment").cast("string"),
        F.col("status").cast("string"),
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),
    )
)

print(
    f"Silver bookings: "
    f"{silver_bookings.count()}"
)

print(
    f"Distinct booking IDs: "
    f"{silver_bookings.select('booking_id').distinct().count()}"
)

In [0]:
(
    silver_bookings.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.silver.fact_booking")
)

In [0]:
%sql
SELECT 'dim_tenant' AS table_name, COUNT(*) AS rows
FROM voyage.silver.dim_tenant

UNION ALL

SELECT 'dim_property', COUNT(*)
FROM voyage.silver.dim_property

UNION ALL

SELECT 'dim_customer', COUNT(*)
FROM voyage.silver.dim_customer

UNION ALL

SELECT 'fact_booking', COUNT(*)
FROM voyage.silver.fact_booking;

In [0]:
%sql
SELECT COUNT(*) AS orphan_bookings
FROM voyage.silver.fact_booking b
LEFT JOIN voyage.silver.dim_customer c
    ON b.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS orphan_bookings
FROM voyage.silver.fact_booking b
LEFT JOIN voyage.silver.dim_property p
    ON b.property_id = p.property_id
WHERE p.property_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS orphan_bookings
FROM voyage.silver.fact_booking b
LEFT JOIN voyage.silver.dim_tenant t
    ON b.tenant_id = t.tenant_id
WHERE t.tenant_id IS NULL;

In [0]:
%sql
SELECT
    SUM(booking_count) AS total_bookings,
    SUM(completed_booking_count) AS completed_bookings,
    SUM(confirmed_booking_count) AS confirmed_bookings,
    SUM(cancelled_booking_count) AS cancelled_bookings,
    SUM(room_nights) AS total_room_nights,
    SUM(estimated_completed_booking_value) AS estimated_completed_value
FROM voyage.dbt_dev.fct_booking_daily;